<div dir="rtl" style="text-align: right;">

# פרויקט למידת מכונה - חלק ב'

בחלק זה בוצעו הכנת נתונים, אימון מודלים והשוואת ביצועים לצורך חיזוי שביעות הרצון של נוסעי חברת התעופה.

לאחר טעינת הנתונים ובדיקת תקינותם, יושמו שלבי הניקוי והעיבוד המקדים בהתאם להחלטות חלק א׳. בהמשך חולק קובץ האימון לסט אימון וסט אימות בחלוקה שכבתית, ונבחנו שני מודלים עיקריים: עץ החלטה ורשת נוירונים מסוג MLP.

 עבור כל מודל בוצע כיוונון היפר־פרמטרים, ולאחר מכן נותחו ביצועיו באמצעות מדדי סיווג, מטריצות בלבול וכלים פרשניים רלוונטיים.

בסיום התהליך בוצעה השוואה בין המודלים על בסיס ביצועיהם בסט האימות, ונבחר המודל המתאים ביותר להגשה לתחרות. המודל הנבחר שימש לחיזוי רמת שביעות הרצון עבור קובץ המבחן הסופי.

</div>


<div dir="rtl" style="text-align: right;">

## 1. ייבוא ספריות והגדרות ראשוניות

נייבא ספריות בסיסיות הדרושות לטעינת הנתונים, בדיקה ראשונית וחלוקה לסט אימון וסט אימות. בהמשך נוסיף ספריות נוספות רק כאשר נגיע לסעיפי המודלים.

</div>


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


RANDOM_STATE = 42

<div dir="rtl" style="text-align: right;">

## 2. טעינת קבצי הנתונים

נטען את קובץ האימון, קובץ המבחן הסופי וקובץ הדוגמה להגשה. בשלב זה קובץ המבחן הסופי משמש רק לבדיקת מבנה, ולא לבחירת מודלים או לכוונון שלהם.

</div>


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DIR = Path.cwd()

TRAIN_FILE = BASE_DIR / "Xy_train.csv"
TEST_FILE = BASE_DIR / "X_test.csv"
EXAMPLE_SUBMISSION_FILE = BASE_DIR / "y test example.xlsx"

required_files = [TRAIN_FILE, TEST_FILE, EXAMPLE_SUBMISSION_FILE]

missing_files = [
    file_path.name
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    print("The following files are missing:")
    print(missing_files)

    if IN_COLAB:
        print("Please select and upload the missing files now:")
        uploaded = files.upload()
    else:
        raise FileNotFoundError(f"The following files are missing from the working directory: {missing_files}")

# בדיקה חוזרת אחרי ההעלאה
missing_files = [
    file_path.name
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(f"The following files are still missing: {missing_files}")

train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
example_submission = pd.read_excel(EXAMPLE_SUBMISSION_FILE)

display(train_raw.head())
display(test_raw.head())
display(example_submission.head())

<div dir="rtl" style="text-align: right;">

## 3. בדיקות מבנה בסיסיות

נבדוק את גודל הקבצים, שמות העמודות, ערכים חסרים וכפילויות. בדיקות אלה נועדו לוודא שאנחנו עובדים עם הקבצים הנכונים לפני שמתחילים לבנות מודלים.

</div>


In [ ]:
data_summary = pd.DataFrame(
    {
        "File": ["Train", "Final Test", "Sample Submission"],
        "Number of Rows": [len(train_raw), len(test_raw), len(example_submission)],
        "Number of Columns": [train_raw.shape[1], test_raw.shape[1], example_submission.shape[1]],
        "Number of Missing Values": [
            int(train_raw.isna().sum().sum()),
            int(test_raw.isna().sum().sum()),
            int(example_submission.isna().sum().sum()),
        ],
        "Number of Duplicate Rows": [
            int(train_raw.duplicated().sum()),
            int(test_raw.duplicated().sum()),
            int(example_submission.duplicated().sum()),
        ],
    }
)

display(data_summary)

In [ ]:
columns_summary = pd.DataFrame(
    {
        "Train Columns": pd.Series(train_raw.columns),
        "Final Test Columns": pd.Series(test_raw.columns),
        "Sample Submission Columns": pd.Series(example_submission.columns),
    }
)

display(columns_summary)

In [ ]:
missing_summary = pd.DataFrame(
    {
        "Missing in Train": train_raw.isna().sum(),
        "Missing in Final Test": test_raw.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_summary)

<div dir="rtl" style="text-align: right;">

## 4. זיהוי עמודת המטרה והפרדת מאפיינים

נזהה את עמודת המטרה מתוך קובץ האימון בפועל. לא נניח מראש את שם העמודה או את צורת הכתיבה שלה. לאחר הזיהוי נפריד בין מאפייני הקלט לבין משתנה המטרה.

</div>


In [ ]:
train_columns = list(train_raw.columns)
test_columns = list(test_raw.columns)

columns_only_in_train = [column for column in train_columns if column not in test_columns]

if len(columns_only_in_train) != 1:
    raise ValueError(
        "The target column could not be identified unambiguously. "
        f"Columns found only in the training file: {columns_only_in_train}"
    )

target_column = columns_only_in_train[0]
feature_columns = [column for column in train_columns if column != target_column]

missing_in_test = [column for column in feature_columns if column not in test_columns]
extra_in_test = [column for column in test_columns if column not in feature_columns]

if missing_in_test or extra_in_test:
    raise ValueError(
        "The feature-column structures of the training and final test files do not match. "
        f"Missing from final test: {missing_in_test}; extra in final test: {extra_in_test}"
    )

X = train_raw[feature_columns].copy()
y = train_raw[target_column].copy()
X_test_final = test_raw[feature_columns].copy()

print(f"Detected target column: {target_column}")
print(f"Number of features: {len(feature_columns)}")

In [ ]:
target_distribution = (
    y.value_counts(dropna=False)
    .rename_axis("Target Value")
    .reset_index(name="Number of Records")
)
target_distribution["Percentage"] = (target_distribution["Number of Records"] / len(y) * 100).round(2)

display(target_distribution)

<div dir="rtl" style="text-align: right;">

## 5. ניקוי נתונים לפי החלטות חלק א'

בשלב זה יושמה לוגיקת הניקוי שעודכנה בעקבות חלק א׳ של הפרויקט. תהליך הניקוי כלל טיפול בערכים חסרים וחריגים, תיקון ערכים לא תקינים, הסרת משתנים שאינם תורמים לחיזוי, והוספת משתנים מעובדים בהתאם להחלטות שהתקבלו בשלב הכנת הנתונים.

בקובץ האימון הוסרו רשומות שלא ניתן להשתמש בהן לצורך למידה מפוקחת, ובפרט רשומות ללא ערך במשתנה המטרה.

לאחר השלמת תהליך הניקוי נותרו 8998 רשומות בקובץ האימון ובקובץ המבחן נשמרו כלל הרשומות (1000) ובסדרן המקורי.
</div>



In [ ]:
service_columns = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Seat comfort",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness",
]

numeric_like_columns = [
    "Age",
    "Flight Distance",
    "Plane colors",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    *service_columns,
]

engineered_features = ["Age_Category", "Flight_Type", "Total_Service_Score"]


def apply_corrected_part_a_cleaning(df_to_clean, is_train=True, target_column_name=None):
    df_clean = df_to_clean.copy()
    original_index = df_clean.index.copy()
    original_rows = len(df_clean)
    summary = {"Rows Before Cleaning": original_rows}

    for column in numeric_like_columns:
        if column in df_clean.columns:
            df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

    # 1. זיהוי והסרת ערכי Class לא תקינים באימון בלבד
    removed_invalid_class_rows = 0
    removed_missing_target_rows = 0
    if "Class" in df_clean.columns:
        valid_classes = ["Eco", "Eco Plus", "Business", "Unknown"]
        missing_class_mask = df_clean["Class"].isna()
        invalid_class_mask = (~df_clean["Class"].isin(valid_classes)) & df_clean["Class"].notna()
        summary["Missing Class Values Replaced"] = int(missing_class_mask.sum())
        summary["Invalid Class Values"] = int(invalid_class_mask.sum())

        if is_train:
            removed_invalid_class_rows = int(invalid_class_mask.sum())
            df_clean = df_clean.drop(index=df_clean.index[invalid_class_mask]).copy()
        else:
            df_clean.loc[invalid_class_mask, "Class"] = np.nan

        # 2. החלפת Class חסר ו-Unknown ב-Business
        missing_class_mask = df_clean["Class"].isna()
        unknown_mask = df_clean["Class"] == "Unknown"
        summary["Unknown Class Values Replaced"] = int(unknown_mask.sum())
        df_clean.loc[missing_class_mask | unknown_mask, "Class"] = "Business"

    # 3. טיפול בערכי Gate location חריגים והשלמה לפי חציון
    if "Gate location" in df_clean.columns:
        invalid_gate_mask = (~df_clean["Gate location"].between(1, 5)) & df_clean["Gate location"].notna()
        summary["Invalid Gate Location Values"] = int(invalid_gate_mask.sum())
        df_clean.loc[invalid_gate_mask, "Gate location"] = np.nan
        gate_median = df_clean["Gate location"].median()
        df_clean["Gate location"] = df_clean["Gate location"].fillna(gate_median)
        summary["Gate Location Imputation Median"] = gate_median

    # 4. השלמת Leg room service לפי Class + Type of Travel, אחר כך Class, אחר כך חציון כללי
    if {"Leg room service", "Class", "Type of Travel"}.issubset(df_clean.columns):
        leg_missing_before = int(df_clean["Leg room service"].isna().sum())
        group_median_1 = df_clean.groupby(["Class", "Type of Travel"])["Leg room service"].transform("median")
        group_median_2 = df_clean.groupby("Class")["Leg room service"].transform("median")
        global_median = df_clean["Leg room service"].median()

        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(group_median_1)
        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(group_median_2)
        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(global_median)

        summary["Missing Leg Room Service Before Imputation"] = leg_missing_before
        summary["Global Leg Room Service Median"] = global_median

    # 5. טיפול בערכי Age חריגים והשלמה לפי חציון
    if "Age" in df_clean.columns:
        invalid_age_mask = ((df_clean["Age"] < 0) | (df_clean["Age"] > 110)) & df_clean["Age"].notna()
        summary["Invalid Age Values"] = int(invalid_age_mask.sum())
        df_clean.loc[invalid_age_mask, "Age"] = np.nan
        age_median = df_clean["Age"].median()
        df_clean["Age"] = df_clean["Age"].fillna(age_median)
        summary["Age Imputation Median"] = age_median

    # 6. טיפול בערכי Flight Distance שליליים והשלמה היררכית
    if {"Flight Distance", "Class", "Type of Travel"}.issubset(df_clean.columns):
        negative_distance_mask = (df_clean["Flight Distance"] < 0) & df_clean["Flight Distance"].notna()
        summary["Negative Flight Distance Values"] = int(negative_distance_mask.sum())
        df_clean.loc[negative_distance_mask, "Flight Distance"] = np.nan

        distance_median_group_1 = df_clean.groupby(["Class", "Type of Travel"])["Flight Distance"].transform("median")
        distance_median_group_2 = df_clean.groupby("Class")["Flight Distance"].transform("median")
        distance_global_median = df_clean["Flight Distance"].median()

        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_1)
        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_2)
        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_global_median)
        summary["Global Flight Distance Median"] = distance_global_median

    # 7. הסרת Plane colors מתוך הטבלה שכבר נוקתה, בלי לחזור לטבלת המקור
    if "Plane colors" in df_clean.columns:
        df_clean = df_clean.drop(columns=["Plane colors"])
        summary["Plane Colors Removed"] = True
    else:
        summary["Plane Colors Removed"] = False

    # לפני השלמה כללית, מסירים באימון בלבד רשומות ללא משתנה מטרה
    if is_train and target_column_name is not None and target_column_name in df_clean.columns:
        missing_target_mask = df_clean[target_column_name].isna()
        removed_missing_target_rows = int(missing_target_mask.sum())
        if removed_missing_target_rows > 0:
            df_clean = df_clean.drop(index=df_clean.index[missing_target_mask]).copy()

    # 8. השלמת חסרים מספריים שנותרו לפי חציון
    numeric_cols = [
        col for col in df_clean.select_dtypes(include=[np.number]).columns
        if col != target_column_name
    ]
    for col in numeric_cols:
        if df_clean[col].isna().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    # 9. השלמת חסרים קטגוריאליים שנותרו לפי שכיח
    categorical_cols = [
        col for col in df_clean.select_dtypes(include=["object"]).columns
        if col != target_column_name
    ]
    for col in categorical_cols:
        if df_clean[col].isna().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

    # 10. יצירת מאפיינים חדשים כמו במחברת המתוקנת
    if "Age" in df_clean.columns:
        df_clean["Age_Category"] = pd.cut(
            df_clean["Age"],
            bins=[0, 12, 18, 65, np.inf],
            labels=["Child", "Teen", "Adult", "Senior"],
            include_lowest=True,
        )

    if "Flight Distance" in df_clean.columns:
        df_clean["Flight_Type"] = pd.cut(
            df_clean["Flight Distance"],
            bins=[0, 1000, 3000, df_clean["Flight Distance"].max()],
            labels=["Short-Haul", "Medium-Haul", "Long-Haul"],
            include_lowest=True,
        )

    available_service_columns = [column for column in service_columns if column in df_clean.columns]
    if available_service_columns:
        df_clean["Total_Service_Score"] = df_clean[available_service_columns].mean(axis=1)

    summary["Rows After Cleaning"] = len(df_clean)
    summary["Class Rows Removed"] = removed_invalid_class_rows
    summary["Rows Removed Due to Missing Target"] = removed_missing_target_rows
    summary["Total Missing Values After Cleaning"] = int(df_clean.isna().sum().sum())
    summary["Engineered Features Created"] = all(feature in df_clean.columns for feature in engineered_features)
    summary["Original Order Preserved"] = df_clean.index.equals(original_index)

    return df_clean, summary


train_for_cleaning = X.copy()
train_for_cleaning[target_column] = y

train_clean, train_cleaning_summary = apply_corrected_part_a_cleaning(
    train_for_cleaning,
    is_train=True,
    target_column_name=target_column,
)

X_test_clean, test_cleaning_summary = apply_corrected_part_a_cleaning(
    X_test_final,
    is_train=False,
    target_column_name=None,
)

X_model = train_clean.drop(columns=[target_column]).copy()
y_model = train_clean[target_column].copy()
X_test_final = X_test_clean.copy()

missing_in_clean_test = [column for column in X_model.columns if column not in X_test_final.columns]
extra_in_clean_test = [column for column in X_test_final.columns if column not in X_model.columns]

if missing_in_clean_test or extra_in_clean_test:
    raise ValueError(
        "The feature structures do not match between training and final test after cleaning. "
        f"Missing from final test: {missing_in_clean_test}; extra in final test: {extra_in_clean_test}"
    )

X_test_final = X_test_final[X_model.columns].copy()

cleaning_summary = pd.DataFrame([train_cleaning_summary, test_cleaning_summary], index=["Train", "Final Test"])
display(cleaning_summary)

print("The data was cleaned within the notebook only. The source files were not modified.")

<div dir="rtl" style="text-align: right;">

### בדיקות לאחר הניקוי

נציג בדיקות שמוודאות שהניקוי בוצע לפי החלטות חלק א', ושקובץ המבחן הסופי נשמר באותו אורך ובאותו סדר.

</div>


In [ ]:
rows_check = pd.DataFrame(
    {
        "File": ["Train", "Final Test"],
        "Rows Before Cleaning": [len(train_raw), len(test_raw)],
        "Rows After Cleaning": [len(train_clean), len(X_test_final)],
        "Rows Removed Due to Missing Target": [train_cleaning_summary["Rows Removed Due to Missing Target"], 0],
    }
)

display(rows_check)

test_order_preserved = X_test_final.index.equals(test_raw.index)
test_row_count_preserved = len(X_test_final) == len(test_raw)

print(f"Final test row count preserved: {test_row_count_preserved}")
print(f"Final test row order preserved: {test_order_preserved}")
print(f"Training rows removed due to missing target: {train_cleaning_summary['Rows Removed Due to Missing Target']}")

missing_after_cleaning = pd.DataFrame(
    {
        "Missing in Training Features After Cleaning": X_model.isna().sum(),
        "Missing in Final Test After Cleaning": X_test_final.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_after_cleaning)

missing_y_after_cleaning = int(y_model.isna().sum())
print(f"Missing values in y after cleaning: {missing_y_after_cleaning}")

plane_colors_removed = ("Plane colors" not in train_clean.columns) and ("Plane colors" not in X_test_final.columns)
engineered_features_created = all(
    feature in train_clean.columns and feature in X_test_final.columns
    for feature in engineered_features
)

print(f"Plane colors removed from training and final test: {plane_colors_removed}")
print(f"Age_Category, Flight_Type, and Total_Service_Score created: {engineered_features_created}")

assert test_row_count_preserved
assert test_order_preserved
assert X_model.isna().sum().sum() == 0
assert missing_y_after_cleaning == 0
assert X_test_final.isna().sum().sum() == 0
assert plane_colors_removed
assert engineered_features_created

<div dir="rtl" style="text-align: right;">

## 6. הכנת נתונים ראשונית לאימון ולאימות

לאחר שלב ניקוי הנתונים, נוספו גם המשתנים ההנדסיים `Age_Category`, `Flight_Type` ו־`Total_Service_Score`, מספר מאפייני הקלט לאחר ההכנה עומד על 23.
</div>


In [ ]:
if y_model.isna().any():
    raise ValueError("Missing values were found in the target column after cleaning. They must be handled before splitting the data.")

original_test_index = X_test_final.index.copy()
original_test_length = len(X_test_final)

print("The cleaned data is ready for the initial training-validation split.")

<div dir="rtl" style="text-align: right;">

## 7. חלוקה לסט אימון וסט אימות

נחלק את קובץ האימון הנקי לסט אימון ולסט אימות. סט האימות יישמר בצד וישמש להערכת ביצועי המודלים על נתונים שלא שימשו לאימון.

החלוקה בוצעה ביחס של 80:20 באמצעות דגימה אקראית-שכבתית לפי משתנה המטרה. לאחר החלוקה התקבלו 7198 רשומות בסט האימון ו- 1800 רשומות בסט האימות.

בדיקת התפלגות משתנה המטרה לאחר החלוקה הראתה שמירה כמעט מלאה על היחס בין המחלקות: בסט האימון 56.36% מהתצפיות הן `neutral or dissatisfied` ו־43.64% הן `satisfied`; בסט האימות 56.33% הן `neutral or dissatisfied` ו־43.67% הן `satisfied`. לכן החלוקה השכבתית שמרה על התפלגות המחלקות בצורה טובה.

</div>


In [ ]:
from sklearn.model_selection import train_test_split

stratify_target = y_model if y_model.nunique(dropna=False) > 1 else None

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

split_summary = pd.DataFrame(
    {
        "Set": ["Train", "Validation", "Final Test"],
        "Number of Records": [len(X_train), len(X_valid), len(X_test_final)],
        "Number of Features": [X_train.shape[1], X_valid.shape[1], X_test_final.shape[1]],
    }
)

display(split_summary)

In [ ]:
train_target_distribution = y_train.value_counts(normalize=True, dropna=False).rename("Train")
valid_target_distribution = y_valid.value_counts(normalize=True, dropna=False).rename("Validation")

split_target_distribution = (
    pd.concat([train_target_distribution, valid_target_distribution], axis=1)
    .fillna(0)
    .mul(100)
    .round(2)
)

display(split_target_distribution)

<div dir="rtl" style="text-align: right;">

### בדיקת תקינות לקובץ המבחן הסופי

בוצעה בדיקה שמספר הרשומות וסדרן בקובץ המבחן הסופי נשמרו לאחר שלבי הניקוי והעיבוד.

</div>

In [ ]:
assert len(X_test_final) == original_test_length
assert X_test_final.index.equals(original_test_index)

print("Final test verification completed: the row count and original order were preserved.")

<div dir="rtl" style="text-align: right;">

## 8. עצי החלטה

בסעיף זה נבנה עץ החלטה מלא, נכוונן היפר-פרמטרים, נציג את העץ הנבחר, ננתח חשיבות משתנים ונעביר רשומת אימות לדוגמה דרך העץ.

</div>

<div dir="rtl" style="text-align: right;">

### 8.1 הכנת הנתונים לעץ החלטה (Data Preparation)

לפני אימון מודל עץ ההחלטה בוצעה התאמה של מבנה הנתונים לדרישות המודל. משתנה המטרה `satisfaction` הומר לערכים בינאריים, כך ש־`neutral or dissatisfied` קודד כ־0 ו־`satisfied` קודד כ־1.

המשתנים חולקו למשתנים מספריים ולמשתנים קטגוריאליים. 17 המשתנים המספריים הועברו למודל ללא סטנדרטיזציה, מכיוון שעצי החלטה מבצעים פיצולים על בסיס ערכי סף של משתנים בודדים ואינם רגישים להבדלי סקאלה בין משתנים.

6 המשתנים הקטגוריאליים, הכוללים את `Gender`, `Customer Type`, `Type of Travel`, `Class`, וכן את המשתנים ההנדסיים `Age_Category` ו־`Flight_Type`, קודדו באמצעות `OneHotEncoder`. קידוד זה מאפשר לייצג משתנים קטגוריאליים כמשתני דמה בינאריים, ללא יצירת סדר מלאכותי בין הקטגוריות.

הטיפול בערכים חסרים וחריגים בוצע במסגרת שלב הניקוי הכללי, ולכן הנתונים שהועברו למודל היו נקיים מערכים חסרים במאפייני הקלט.

</div>


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
import numpy as np
import pandas as pd

# 1. מיפוי משתנה המטרה לבינארי
TARGET_MAP = {
    'neutral or dissatisfied': 0,
    'satisfied': 1
}

y_train_dt = y_train.map(TARGET_MAP).astype(int)
y_valid_dt = y_valid.map(TARGET_MAP).astype(int)

print('Target variable mapped successfully.')

In [ ]:
# 2. זיהוי עמודות מספריות וקטגוריאליות
numeric_features_dt = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_dt = [col for col in X_train.columns if col not in numeric_features_dt]

def make_dt_preprocessor():
    try:
        one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

    return ColumnTransformer(
        transformers=[
            # מותירים את המשתנים המספריים ללא שינוי (passthrough) וללא StandardScaler
            ('numeric', 'passthrough', numeric_features_dt),
            ('categorical', one_hot_encoder, categorical_features_dt),
        ],
        remainder='drop',
    )

def build_dt_pipeline(model):
    return Pipeline(
        steps=[
            ('preprocess', make_dt_preprocessor()),
            ('model', model),
        ]
    )

dt_preprocessing_summary = pd.DataFrame(
    {
        'Variable Type': ['Numeric', 'Categorical'],
        'Number of Variables': [len(numeric_features_dt), len(categorical_features_dt)],
        'Preprocessing Operation': ['Passthrough (no change)', 'One-Hot Encoding'],
    }
)
display(dt_preprocessing_summary)

<div dir="rtl" style="text-align: right;">

### 8.2 בניית עץ החלטה מלא (Full Decision Tree)

נאמן עץ החלטה מלא ללא מגבלת עומק או פיצול (ערכי ברירת מחדל של `DecisionTreeClassifier` מלבד ה-`random_state`). נחשב את מדדי הביצוע על סט האימון וסט האימות, ונציג אותם בגרף עמודות.

</div>


In [ ]:
# עץ החלטה מלא - אימון, הערכה והצגת תוצאות

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. בניית מודל עץ החלטה מלא
# לא מגבילים את עומק העץ או את מספר הפיצולים, כדי לבדוק איך עץ מלא מתנהג
full_dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)

# 2. חיבור המודל לצינור העיבוד המקדים
# הצינור כולל את שלב עיבוד הנתונים ואת המודל עצמו
full_dt_pipeline = build_dt_pipeline(full_dt_model)

# 3. אימון המודל על סט האימון
full_dt_pipeline.fit(X_train, y_train_dt)


# 4. פונקציה להערכת ביצועי המודל ולהצגת גרפים
def evaluate_full_decision_tree(model_name, fitted_pipeline, return_metrics=False):

    # נשמור כאן את מדדי הביצוע של המודל
    metrics = []

    # נבדוק את המודל גם על סט האימון וגם על סט האימות
    splits = [
        ('Train', X_train, y_train_dt),
        ('Validation', X_valid, y_valid_dt)
    ]

    # חישוב מדדי הביצוע לכל סט נתונים
    for split_name, X_data, y_true in splits:

        # חיזוי ערכי המטרה באמצעות המודל
        y_pred = fitted_pipeline.predict(X_data)

        # חישוב המדדים ושמירתם בטבלה
        metrics.append({
            'Dataset': split_name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0)
        })

    # יצירת טבלת ביצועים
    df_metrics = pd.DataFrame(metrics)

    # רשימת המדדים שנרצה להציג
    metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1']

    # חילוץ ערכי המדדים עבור סט האימון וסט האימות
    train_values = df_metrics.loc[df_metrics['Dataset'] == 'Train', metric_cols].iloc[0]
    valid_values = df_metrics.loc[df_metrics['Dataset'] == 'Validation', metric_cols].iloc[0]

    # יצירת טבלת פערים בין ביצועי האימון לביצועי האימות
    # פער גדול מצביע על חשש להתאמת יתר
    gap_df = pd.DataFrame({
        'Metric': metric_cols,
        'Train': train_values.values,
        'Validation': valid_values.values,
        'Gap (Train - Validation)': train_values.values - valid_values.values
    })

    # -------------------------------------------------
    # גרף 1: השוואת ביצועים בין סט האימון לסט האימות
    # -------------------------------------------------

    x = np.arange(len(metric_cols))
    width = 0.35

    plt.figure(figsize=(9, 5))

    train_bars = plt.bar(
        x - width / 2,
        gap_df['Train'],
        width,
        label='Train'
    )

    valid_bars = plt.bar(
        x + width / 2,
        gap_df['Validation'],
        width,
        label='Validation'
    )

    plt.title(f'{model_name} - Performance Metrics')
    plt.xlabel('Metric')
    plt.ylabel('Score')
    plt.xticks(x, metric_cols)
    plt.ylim(0, 1.05)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # הוספת ערכים מספריים מעל העמודות
    for bars in [train_bars, valid_bars]:
        for bar in bars:
            height = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f'{height:.3f}',
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------
    # גרף 2: פער ההכללה בין סט האימון לסט האימות
    # -------------------------------------------------

    plt.figure(figsize=(8, 4))

    gap_bars = plt.bar(
        gap_df['Metric'],
        gap_df['Gap (Train - Validation)']
    )

    plt.title(f'{model_name} - Generalization Gap')
    plt.xlabel('Metric')
    plt.ylabel('Train - Validation Gap')
    plt.ylim(0, gap_df['Gap (Train - Validation)'].max() + 0.05)
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # הוספת ערכים מספריים מעל העמודות
    for bar in gap_bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.005,
            f'{height:.3f}',
            ha='center',
            va='bottom',
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

    # הצגת טבלאות הביצועים והפערים
    print("Performance metrics:")
    display(df_metrics)

    print("Generalization gap:")
    display(gap_df)

    # החזרת הטבלאות לשימוש בהמשך המחברת
    if return_metrics:
        return df_metrics, gap_df


# 5. הרצת פונקציית ההערכה עבור עץ ההחלטה המלא
full_dt_metrics, full_dt_gap = evaluate_full_decision_tree(
    model_name='Full Decision Tree',
    fitted_pipeline=full_dt_pipeline,
    return_metrics=True
)

<div dir="rtl" style="text-align: right;">

**מסקנות מעץ מלא:**

עץ ההחלטה המלא הגיע לדיוק של 100% על סט האימון: Accuracy, Precision, Recall ו־F1 כולם שווים ל־1.0000. לעומת זאת, על סט האימות התקבלו ביצועים נמוכים יותר: Accuracy=0.8478, Precision=0.8216, Recall=0.8321 ו־F1=0.8268.

הפער בין ביצועי האימון לביצועי האימות מצביע על התאמת יתר ברורה. כלומר, העץ למד בצורה טובה מאוד את נתוני האימון, אך חלק מהלמידה הייתה ספציפית מדי לרעש או לפרטים נקודתיים בסט האימון ולכן אינה מכלילה באותה רמה לנתונים חדשים.

עץ מלא לא תמיד יגיע ל־100% דיוק על סט האימון. אם קיימות בנתונים רשומות בעלות מאפיינים זהים לחלוטין אך תגיות מטרה שונות, גם עץ מלא לא יוכל להפריד ביניהן באופן מושלם. במקרה שלנו, העץ הצליח להגיע לדיוק מלא על סט האימון, ולכן הדבר מחזק את החשש להתאמת יתר.

<div dir="rtl" style="text-align: right;">

### 8.3 כיוונון היפר-פרמטרים (Hyperparameter Tuning)

נכוונן היפר-פרמטרים כדי לשפר את יכולת ההכללה של המודל ולמנוע התאמת יתר. נבצע כיוונון עבור ארבעה היפר-פרמטרים מרכזיים:
* **`max_depth` (עומק מקסימלי):** מגביל את כמות הפיצולים הרציפה לאורך העץ. הגדלת הערך מאפשרת לעץ ללמוד חוקיות מורכבת יותר (מגדילה את מורכבות העץ ויכולה להוביל להתאמת יתר), והקטנתו מרסנת את העץ ומפחיתה מורכבות (יכולה להוביל לתת-התאמה).
* **`ccp_alpha` (גיזום עלות-מורכבות):** שולט על גודל העץ באמצעות הענשת מורכבותו (עלי העץ). ערך אלפא גדול יותר מביא לגיזום משמעותי יותר (עץ קטן ופשוט יותר עם פחות צמתים), בעוד שערך אלפא שואף ל-0 מאפשר לעץ לגדול ללא הפרעה.
* **`min_samples_split` (מינימום דוגמאות לפיצול צומת):** מונע מצמתים קטנים מלהתפצל הלאה, מה שמקטין את הרגישות לרעש בנתונים. הגדלת הערך מונעת פיצולים ספציפיים ומרסנת את העץ, בעוד ערך קטן מאפשר לעץ להתאים את עצמו גם לצמתים קטנים מאוד.
* **`min_samples_leaf` (מינימום דוגמאות בעלה):** מבטיח שכל עלה יכיל מספר מינימלי של דוגמאות, ובכך מונע החלטות המבוססות על דוגמאות בודדות ומקריות. הגדלת הערך מגבירה את ההחלקה של המודל ומפחיתה את הסיכון להתאמת יתר.


נשתמש בשילוב של שתי גישות:
* **חיפוש 1D (Sweeps):** נבצע הרצות נפרדות לכל אחד מההיפר-פרמטרים על מנת להציג את השפעתם בנפרד (כולל עקומות דיוק אימון מול אימות). עבור `ccp_alpha` נשתמש במסלול הגיזום המתקבל מ-`cost_complexity_pruning_path`, שהוא פתרון מתמטי אופטימלי ויעיל במיוחד.
* **חיפוש 4D Grid Search:** לבסוף, נבצע סריקת רשת (Grid Search) משולבת על פני ארבעת הממדים יחד על מנת למצוא את השילוב המנצח של הפרמטרים שיוביל לביצועים המיטביים על סט האימות.

</div>

In [ ]:
# 1. יצירת רשת חיפוש לעומק העץ
max_depth_list = np.arange(1, 16, 1)
preprocessor = make_dt_preprocessor()
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_valid_preprocessed = preprocessor.transform(X_valid)

train_accs = []
valid_accs = []

for depth in max_depth_list:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    clf.fit(X_train_preprocessed, y_train_dt)
    train_accs.append(accuracy_score(y_train_dt, clf.predict(X_train_preprocessed)))
    valid_accs.append(accuracy_score(y_valid_dt, clf.predict(X_valid_preprocessed)))

# 2. ציור גרף דיוק כפונקציה של max_depth
plt.figure(figsize=(9, 4.5))
plt.plot(max_depth_list, train_accs, marker='o', label='Train Accuracy', linestyle='--')
plt.plot(max_depth_list, valid_accs, marker='o', label='Validation Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Max Depth (Pre-pruning)')
plt.xticks(max_depth_list)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# זיהוי העומק המיטבי ב-pre-pruning
best_depth_idx = np.argmax(valid_accs)
best_depth = max_depth_list[best_depth_idx]
print(f'Optimal max_depth according to pre-pruning: {best_depth} (Validation Accuracy: {valid_accs[best_depth_idx]:.4f})')

<div dir="rtl" style="text-align: right;">

בסריקה החד־ממדית של `max_depth` התקבל כי עומק 8 מספק את דיוק האימות הגבוה ביותר, עם Validation Accuracy=0.8761. ניתן לראות כי ככל שהעומק גדל, דיוק האימון ממשיך לעלות, אך דיוק האימות נעצר ואף מתחיל לרדת מעט. תוצאה זו מצביעה על כך שעומק גבוה מדי גורם לעץ ללמוד דפוסים נקודתיים מדי ולהיכנס להתאמת יתר.

In [ ]:
# 1. הפקת מסלול גיזום עלות-מורכבות (Cost Complexity Path)
clf_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
clf_full.fit(X_train_preprocessed, y_train_dt)
path = clf_full.cost_complexity_pruning_path(X_train_preprocessed, y_train_dt)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# 2. ציור גרף Impurity vs Alpha
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker='o', drawstyle='steps-post')
ax.set_xlabel('Effective Alpha (ccp_alpha)')
ax.set_ylabel('Total Impurity of Leaves')
ax.set_title('Total Impurity vs Alpha for Training Set')
ax.grid(alpha=0.3)
plt.show()

<div dir="rtl" style="text-align: right;">

הגרף מציג את מסלול גיזום עלות־מורכבות של עץ ההחלטה. ככל שערך `ccp_alpha` גדל, מתבצע גיזום חזק יותר של העץ: מספר הענפים והעלים קטן, והמודל הופך פשוט יותר. במקביל, סך אי־הטוהר בעלים עולה, משום שעץ פשוט יותר מבצע פחות פיצולים ולכן מתאים פחות במדויק לנתוני האימון.

גרף זה מתאר את השפעת הגיזום על מבנה העץ, אך אינו קובע לבדו את ערך `ccp_alpha` המיטבי מבחינת ביצועי אימות.

</div>

In [ ]:
# 1. אימון עצי החלטה עבור כל ערך ccp_alpha במסלול (נשמיט את האחרון המייצג עץ עם עלה בודד)
clfs = []
ccp_alphas_subset = ccp_alphas[:-1]
for ccp_alpha in ccp_alphas_subset:
    clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=ccp_alpha)
    clf.fit(X_train_preprocessed, y_train_dt)
    clfs.append(clf)

# 2. חילוץ כמות צמתים ועומק לכל עץ
node_counts = [clf.tree_.node_count for clf in clfs]
depths = [clf.tree_.max_depth for clf in clfs]

# 3. ציור גרפי מורכבות העץ
fig, ax = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
ax[0].plot(ccp_alphas_subset, node_counts, marker='o', drawstyle='steps-post', color='purple')
ax[0].set_ylabel('Number of Nodes')
ax[0].set_title('Number of Nodes vs Alpha')
ax[0].grid(alpha=0.3)

ax[1].plot(ccp_alphas_subset, depths, marker='o', drawstyle='steps-post', color='green')
ax[1].set_xlabel('Effective Alpha (ccp_alpha)')
ax[1].set_ylabel('Depth of Tree')
ax[1].set_title('Depth vs Alpha')
ax[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right;">

הגרף הבא מציג את מסלול הגיזום של העץ. ככל שערך `ccp_alpha` גדל, מתבצע גיזום חזק יותר, מספר הצמתים קטן, והעץ הופך פשוט יותר.

In [ ]:
# 1. חישוב ביצועי דיוק עבור כל ccp_alpha
train_accs_ccp = [accuracy_score(y_train_dt, clf.predict(X_train_preprocessed)) for clf in clfs]
valid_accs_ccp = [accuracy_score(y_valid_dt, clf.predict(X_valid_preprocessed)) for clf in clfs]

# 2. ציור גרף Accuracy vs Alpha
plt.figure(figsize=(9, 4.5))
plt.plot(ccp_alphas_subset, train_accs_ccp, marker='o', label='Train Accuracy', drawstyle='steps-post')
plt.plot(ccp_alphas_subset, valid_accs_ccp, marker='o', label='Validation Accuracy', drawstyle='steps-post')
plt.xlabel('Effective Alpha (ccp_alpha)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Alpha for Train and Validation Sets')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# זיהוי ה-ccp_alpha המיטבי
best_ccp_idx = np.argmax(valid_accs_ccp)
best_ccp_alpha = ccp_alphas_subset[best_ccp_idx]
print(f'Optimal ccp_alpha according to validation accuracy: {best_ccp_alpha:.6f} (Validation Accuracy: {valid_accs_ccp[best_ccp_idx]:.4f})')

<div dir="rtl" style="text-align: right;">

בסריקה החד־ממדית של `ccp_alpha` התקבל הערך `ccp_alpha=0.000507`, עם דיוק אימות של `0.8883`. ערך זה שיפר את ביצועי האימות ביחס לעץ המלא, ולכן מצביע על כך שגיזום מתון סייע להפחתת התאמת היתר.

עם זאת, בשלב זה נבחן רק פרמטר אחד בנפרד. בהמשך בוצעה סריקה משולבת של מספר היפר־פרמטרים, ולכן תוצאה זו שימשה כאינדיקציה ראשונית ולא כקונפיגורציה הסופית של המודל.

</div>

In [ ]:
# ==============================================================================
# כיוונון 1D עבור min_samples_split ו-min_samples_leaf
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------------------------
# סריקת 1D עבור min_samples_split
# ------------------------------------------------------------------------------
min_samples_split_list = np.arange(2, 101, 5)
train_accs_split = []
valid_accs_split = []

for split in min_samples_split_list:
    clf = DecisionTreeClassifier(min_samples_split=split, random_state=RANDOM_STATE)
    clf.fit(X_train_preprocessed, y_train_dt)
    train_accs_split.append(accuracy_score(y_train_dt, clf.predict(X_train_preprocessed)))
    valid_accs_split.append(accuracy_score(y_valid_dt, clf.predict(X_valid_preprocessed)))

plt.figure(figsize=(9, 4.5))
plt.plot(min_samples_split_list, train_accs_split, marker='o', label='Train Accuracy', linestyle='--')
plt.plot(min_samples_split_list, valid_accs_split, marker='o', label='Validation Accuracy')
plt.xlabel('Min Samples Split')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Min Samples Split (1D Sweep)')
plt.xticks(np.arange(0, 101, 10))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_split_idx = np.argmax(valid_accs_split)
best_split_val = min_samples_split_list[best_split_idx]
print(f'Optimal min_samples_split according to 1D sweep: {best_split_val} (Validation Accuracy: {valid_accs_split[best_split_idx]:.4f})')

# ------------------------------------------------------------------------------
# סריקת 1D עבור min_samples_leaf
# ------------------------------------------------------------------------------
min_samples_leaf_list = np.arange(1, 51, 2)
train_accs_leaf = []
valid_accs_leaf = []

for leaf in min_samples_leaf_list:
    clf = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=RANDOM_STATE)
    clf.fit(X_train_preprocessed, y_train_dt)
    train_accs_leaf.append(accuracy_score(y_train_dt, clf.predict(X_train_preprocessed)))
    valid_accs_leaf.append(accuracy_score(y_valid_dt, clf.predict(X_valid_preprocessed)))

plt.figure(figsize=(9, 4.5))
plt.plot(min_samples_leaf_list, train_accs_leaf, marker='o', label='Train Accuracy', linestyle='--')
plt.plot(min_samples_leaf_list, valid_accs_leaf, marker='o', label='Validation Accuracy')
plt.xlabel('Min Samples Leaf')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Min Samples Leaf (1D Sweep)')
plt.xticks(np.arange(0, 51, 5))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_leaf_idx = np.argmax(valid_accs_leaf)
best_leaf_val = min_samples_leaf_list[best_leaf_idx]
print(f'Optimal min_samples_leaf according to 1D sweep: {best_leaf_val} (Validation Accuracy: {valid_accs_leaf[best_leaf_idx]:.4f})')

<div dir="rtl" style="text-align: right;">

* **עקומת `min_samples_split`:** עבור ערכים נמוכים דיוק האימון גבוה מאוד, אך דיוק האימות מגיע לשיא סביב `min_samples_split=77`, עם Validation Accuracy=0.8822, ולאחר מכן נותר יציב יחסית או יורד מעט. הגדלת הערך מרסנת את העץ ומפחיתה התאמת יתר.
* **עקומת `min_samples_leaf`:** שיא דיוק האימות התקבל סביב `min_samples_leaf=29`, עם Validation Accuracy=0.8900. ערך זה מונע עלים קטנים מדי ולכן תורם למודל יציב ומוכלל יותר.

</div>

In [ ]:
# ------------------------------------------------------------------------------
# הרצת 4D Grid Search משולב
# ------------------------------------------------------------------------------
depth_options = [6, 8, 9, 10, 12]
# נפיק את ערכי ccp_alpha האפקטיביים ונבחר 10 מתוכם
clf_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
clf_full.fit(X_train_preprocessed, y_train_dt)
path = clf_full.cost_complexity_pruning_path(X_train_preprocessed, y_train_dt)
ccp_alphas_options = path.ccp_alphas[:-1]
sampled_alphas = np.unique(np.percentile(ccp_alphas_options, np.linspace(0, 100, 10)))

min_samples_split_options = [30, 60, 80, 100]
min_samples_leaf_options = [15, 25, 35, 45]

best_combined_acc = 0
best_combined_depth = None
best_combined_alpha = None
best_combined_split = None
best_combined_leaf = None

print("\nRunning 4D Grid Search...")
for depth in depth_options:
    for alpha in sampled_alphas:
        for split in min_samples_split_options:
            for leaf in min_samples_leaf_options:
                clf = DecisionTreeClassifier(
                    max_depth=depth,
                    ccp_alpha=alpha,
                    min_samples_split=split,
                    min_samples_leaf=leaf,
                    random_state=RANDOM_STATE
                )
                clf.fit(X_train_preprocessed, y_train_dt)
                val_acc = accuracy_score(y_valid_dt, clf.predict(X_valid_preprocessed))

                if val_acc > best_combined_acc:
                    best_combined_acc = val_acc
                    best_combined_depth = depth
                    best_combined_alpha = alpha
                    best_combined_split = split
                    best_combined_leaf = leaf

print("="*50)
print("Best combined result from the 4D Grid Search:")
print(f"Optimal max_depth: {best_combined_depth}")
print(f"Optimal ccp_alpha: {best_combined_alpha:.6f}")
print(f"Optimal min_samples_split: {best_combined_split}")
print(f"Optimal min_samples_leaf: {best_combined_leaf}")
print(f"Validation Accuracy: {best_combined_acc:.4f}")
print("="*50)

<div dir="rtl" style="text-align: right;">

**מסקנות מהסריקה המשולבת:**

בסריקה המשולבת נבחנו יחד ארבעת ההיפר־פרמטרים המרכזיים של העץ: `max_depth`, `ccp_alpha`, `min_samples_split` ו־`min_samples_leaf`. בניגוד לסריקות החד־ממדיות, שבהן כל פרמטר נבחן בנפרד, הסריקה המשולבת מאפשרת למצוא שילוב מאוזן בין כמה מנגנוני ריסון של העץ.

הקונפיגורציה המיטבית שהתקבלה היא:
`max_depth = 9`, `ccp_alpha = 0.0`, `min_samples_split = 30`, `min_samples_leaf = 25`.

דיוק האימות שהתקבל עבור שילוב זה הוא Validation Accuracy=0.8917. כלומר, בסופו של דבר לא נבחר גיזום לפי `ccp_alpha`, אלא שילוב של הגבלת עומק, מינימום דוגמאות לפיצול ומינימום דוגמאות בעלה. שילוב זה מרסן את העץ ומפחית התאמת יתר גם ללא גיזום עלות־מורכבות.

</div>

<div dir="rtl" style="text-align: right;">


### 8.4 הרצת עץ ההחלטה המיטבי וניתוח התוצאות

עץ ההחלטה המיטבי אומן לפי שילוב הפרמטרים האופטימלי שהתקבל מסריקת הרשת המשולבת (4D Grid Search), ולאחר מכן נבחנו ביצועיו על סט האימון ועל סט האימות בהשוואה לעץ ההחלטה המלא.


</div>

In [ ]:
# 1. אימון עץ מיטבי בעזרת הפרמטרים האופטימליים מסריקת הרשת המשולבת
best_dt_model = DecisionTreeClassifier(
    max_depth=best_combined_depth,
    ccp_alpha=best_combined_alpha,
    min_samples_split=best_combined_split,
    min_samples_leaf=best_combined_leaf,
    random_state=RANDOM_STATE
)
best_dt_pipeline = build_dt_pipeline(best_dt_model)
best_dt_pipeline.fit(X_train, y_train_dt)

# 2. חישוב מדדי ביצוע על סט האימון וסט האימות
best_dt_metrics = pd.DataFrame([
    {
        'Dataset': 'Train',
        'Accuracy': accuracy_score(y_train_dt, best_dt_pipeline.predict(X_train)),
        'Precision': precision_score(y_train_dt, best_dt_pipeline.predict(X_train), zero_division=0),
        'Recall': recall_score(y_train_dt, best_dt_pipeline.predict(X_train), zero_division=0),
        'F1': f1_score(y_train_dt, best_dt_pipeline.predict(X_train), zero_division=0)
    },
    {
        'Dataset': 'Validation',
        'Accuracy': accuracy_score(y_valid_dt, best_dt_pipeline.predict(X_valid)),
        'Precision': precision_score(y_valid_dt, best_dt_pipeline.predict(X_valid), zero_division=0),
        'Recall': recall_score(y_valid_dt, best_dt_pipeline.predict(X_valid), zero_division=0),
        'F1': f1_score(y_valid_dt, best_dt_pipeline.predict(X_valid), zero_division=0)
    }
])

print("Optimized pruned decision tree performance:")
display(best_dt_metrics.round(4))

# 3. גרף השוואתי של מדדי הביצוע
best_dt_plot = best_dt_metrics.melt(
    id_vars='Dataset',
    var_name='Metric',
    value_name='Score'
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=best_dt_plot,
    x='Metric',
    y='Score',
    hue='Dataset',
    palette='muted'
)

# הוספת ערכים מעל כל עמודה
for p in plt.gca().patches:
    height = p.get_height()
    if height > 0:
        plt.gca().annotate(
            f'{height:.3f}',
            (p.get_x() + p.get_width() / 2, height),
            ha='center',
            va='bottom',
            fontsize=9,
            xytext=(0, 3),
            textcoords='offset points'
        )

plt.title('Optimized Pruned Decision Tree - Performance Metrics')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Dataset')
plt.tight_layout()
plt.show()

# 4. השוואה מספרית בין העץ המלא לעץ המגוזם
comparison_df = pd.concat([
    full_dt_metrics.assign(Model='Full Decision Tree'),
    best_dt_metrics.assign(Model='Optimized Pruned Decision Tree')
])

print("Comparison between full and optimized pruned decision trees:")
display(comparison_df.round(4))

<div dir="rtl" style="text-align: right;">

העץ המיטבי שהתקבל לאחר הסריקה המשולבת הגיע ל־Accuracy=0.9036 על סט האימון ול־Accuracy=0.8917 על סט האימות. בנוסף, על סט האימות התקבלו Precision=0.8863, Recall=0.8626 ו־F1=0.8743.

בהשוואה לעץ המלא, ביצועי האימון ירדו מ־1.0000 ל־0.9036, אך ביצועי האימות השתפרו מ־0.8478 ל־0.8917. כלומר, המודל הסופי פחות “זוכר” את נתוני האימון, אך מכליל טוב יותר לנתונים חדשים. פער ההכללה קטן משמעותית, ולכן ניתן להסיק שהכיוונון הפחית התאמת יתר.
</div>

<div dir="rtl" style="text-align: right;">

**תובנות ממבנה העץ וחשיבות המאפיינים:**

צומת השורש של העץ הוא `Type of Travel_Personal Travel <= 0.5`, כלומר הפיצול הראשון מתבצע לפי סוג הנסיעה. מאחר שזהו הפיצול הראשון בעץ, ניתן להסיק שסוג הנסיעה הוא אחד הגורמים המרכזיים ביותר בהבחנה בין נוסעים מרוצים לבין נוסעים שאינם מרוצים.

עבור נוסעים בנסיעת עסקים, כלומר כאשר `Personal Travel=0`, הרשומות עוברות לענף השמאלי. בענף זה ההחלטות הבאות מתבססות בעיקר על `Total_Service_Score`, ולאחר מכן על משתני שירות כמו `Inflight wifi service`, סוג הלקוח ורמת הניקיון.

עבור נוסעים בנסיעה פרטית, כלומר כאשר `Personal Travel=1`, הרשומות עוברות לענף הימני. בענף זה הפיצול המרכזי הבא מתבסס על `Inflight wifi service`, ובהמשך גם על `Ease of Online booking`.

מבנה זה מלמד כי המודל מבסס את החלטותיו בעיקר על שילוב בין הקשר הנסיעה לבין איכות חוויית השירות. משתנים המופיעים גבוה בעץ משפיעים מוקדם יותר על תהליך הסיווג ולכן מספקים אינדיקציה לחשיבותם במודל.

</div>

In [ ]:
# 1. ציור גרף של מבנה העץ המגוזם
# מציגים עד עומק 3
preprocessed_feature_names = best_dt_pipeline.named_steps["preprocess"].get_feature_names_out()

plt.figure(figsize=(22, 6), dpi=200)

plot_tree(
    best_dt_pipeline.named_steps["model"],
    max_depth=3,
    feature_names=preprocessed_feature_names,
    class_names=["Neutral/Dissatisfied", "Satisfied"],
    filled=True,
    rounded=True,
    fontsize=8,
    impurity=False,
    proportion=False,
    precision=2
)

plt.title("Best Decision Tree Structure (Pruned, Max Depth 3 Shown)", fontsize=14)
plt.tight_layout()
plt.show()


# 2. ציור מטריצת בלבול
best_dt_valid_predictions = best_dt_pipeline.predict(X_valid)

plt.figure(figsize=(6, 5), dpi=150)
ConfusionMatrixDisplay.from_predictions(
    y_valid_dt,
    best_dt_valid_predictions,
    display_labels=["Neutral/Dissatisfied", "Satisfied"],
    cmap="Blues",
    values_format="d"
)
plt.title("Best Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# מעבר ידני של רשומה לדוגמה בעץ ההחלטה המיטבי

# 1. בחירת רשומה מסט האימות
sample_idx = 10  # ניתן לשנות אינדקס אם רוצים לבחור רשומה אחרת
sample_features = X_valid.iloc[[sample_idx]]
sample_true_label = y_valid_dt.iloc[sample_idx]

# 2. הצגת נתוני הרשומה שנבחרה
display(sample_features.T.rename(columns={sample_features.index[0]: "Feature Value"}))

print(f"True Label: {sample_true_label} ({'Satisfied' if sample_true_label == 1 else 'Neutral/Dissatisfied'})")

# 3. חיזוי המודל עבור הרשומה
sample_pred = best_dt_pipeline.predict(sample_features)[0]

print(f"Model Prediction: {sample_pred} ({'Satisfied' if sample_pred == 1 else 'Neutral/Dissatisfied'})")

# 4. חילוץ שלב העיבוד המקדים והמודל מתוך ה-Pipeline
preprocessor = best_dt_pipeline.named_steps['preprocess']
tree_model = best_dt_pipeline.named_steps['model']

# 5. עיבוד הרשומה באותו אופן שבו המודל מקבל אותה
sample_processed = preprocessor.transform(sample_features)

# 6. חילוץ מסלול ההחלטה בעץ
node_indicator = tree_model.decision_path(sample_processed)
leaf_id = tree_model.apply(sample_processed)[0]
tree = tree_model.tree_

print("\nDecision path for selected sample:")

for node_id in node_indicator.indices:

    # עצירה כאשר מגיעים לעלה הסופי
    if node_id == leaf_id:
        print(f"Reached leaf node {node_id}.")
        break

    feature_idx = tree.feature[node_id]
    threshold = tree.threshold[node_id]

    feature_name = preprocessed_feature_names[feature_idx]
    feature_value = sample_processed[0, feature_idx]

    if feature_value <= threshold:
        direction = "left"
        condition = "<="
    else:
        direction = "right"
        condition = ">"

    print(
        f"Node {node_id}: {feature_name} = {feature_value:.3f} "
        f"{condition} {threshold:.3f} → go {direction}"
    )

# 7. סיכום
print("\nSummary for report:")
print("Position in validation set:", sample_idx)
print("Original row index:", sample_features.index[0])
print(f"Prediction: {'Satisfied' if sample_pred == 1 else 'Neutral/Dissatisfied'}")
print(f"True value: {'Satisfied' if sample_true_label == 1 else 'Neutral/Dissatisfied'}")
print(f"Correct classification: {'Yes' if sample_pred == sample_true_label else 'No'}")



<div dir="rtl" style="text-align: right;">

**תיאור מעבר ידני על פי גרף העץ עבור הרשומה שנבחרה:**

לצורך המחשת אופן פעולת המודל, נבחרה הרשומה במיקום 10 מתוך סט האימות, כאשר האינדקס המקורי שלה הוא 4267. מדובר בנוסע זכר, לקוח נאמן, בן 38, בנסיעת עסקים, במחלקת Business. בין מאפייניו הבולטים: `Inflight wifi service=3`, `Cleanliness=4`, `Seat comfort=1`, ו־`Total_Service_Score=2.75`.

מסלול ההחלטה של הרשומה בעץ היה:
1. בצומת 0 נבדק התנאי `Type of Travel_Personal Travel <= 0.500`. מכיוון שמדובר בנסיעת עסקים, ערך המשתנה הוא 0 ולכן התנאי מתקיים והרשומה עוברת שמאלה.
2. בצומת 1 נבדק התנאי `Total_Service_Score <= 2.750`. ערך הרשומה הוא 2.75 ולכן התנאי מתקיים והרשומה עוברת שמאלה.
3. בצומת 2 נבדק התנאי `Inflight wifi service <= 3.000`. ערך הרשומה הוא 3 ולכן התנאי מתקיים והרשומה עוברת שמאלה.
4. בצומת 4 נבדק התנאי `Customer Type_Loyal Customer <= 0.500`. מכיוון שהנוסע הוא לקוח נאמן, ערך המשתנה הוא 1 ולכן התנאי אינו מתקיים והרשומה עוברת ימינה.
5. בהמשך המסלול נבדקו תנאים נוספים על `Cleanliness`, `Inflight service`, `Seat comfort` ו־`Age`, עד שהרשומה הגיעה לעלה 59.
6. העלה הסופי סיווג את הרשומה כ־`Satisfied`.

תחזית המודל הייתה `Satisfied`, וגם הערך האמיתי של הרשומה היה `Satisfied`. לכן במקרה זה הסיווג של המודל תאם את המציאות.
</div>


In [ ]:
# 1. חילוץ חשיבות המשתנים ממודל עץ ההחלטה המיטבי
importances = best_dt_model.feature_importances_
indices = np.argsort(importances)[::-1]

# 2. יצירת טבלת חשיבות משתנים, ממוינת מהמשתנה החשוב ביותר לפחות חשוב
feature_importance_df = pd.DataFrame({
    "Feature": preprocessed_feature_names[indices],
    "Importance": importances[indices]
})

# 3. גרף 15 המשתנים החשובים ביותר
top_15_features = feature_importance_df.head(15)

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    x="Importance",
    y="Feature",
    data=top_15_features
)

ax.set_title("Top 15 Feature Importances (Best Decision Tree)")
ax.set_xlabel("Relative Importance")
ax.set_ylabel("Feature")

# הוספת ערכי החשיבות ליד כל עמודה
for p in ax.patches:
    width = p.get_width()
    ax.text(
        width + 0.005,
        p.get_y() + p.get_height() / 2,
        f"{width:.4f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

# 4. טבלת 5 המשתנים החשובים ביותר עבור הדוח
top_5_feature_importance = feature_importance_df.head(5).reset_index(drop=True)
top_5_feature_importance.insert(0, "Rank", range(1, 6))

print("Top 5 feature importances for Table 4:")
display(top_5_feature_importance.round(4).style.hide(axis="index"))

<div dir="rtl" style="text-align: right;">

**ניתוח חשיבות משתנים והשוואה לחלק א':**

פונקציית `feature_importances_` מחשבת את התרומה היחסית של כל משתנה להפחתת אי־הטוהר בצמתי העץ. ככל שמשתנה תורם יותר לפיצולים שמפחיתים את אי־הטוהר, כך ערך החשיבות שלו גבוה יותר.

חמשת המשתנים החשובים ביותר בעץ ההחלטה המיטבי היו:
1. `Type of Travel_Personal Travel` עם חשיבות 0.2739
2. `Total_Service_Score` עם חשיבות 0.1845
3. `Inflight wifi service` עם חשיבות 0.1844
4. `Customer Type_disloyal Customer` עם חשיבות 0.0975
5. `Cleanliness` עם חשיבות 0.0848

הממצאים מתיישבים עם התובנות שעלו בחלק א׳: שביעות הרצון מושפעת בעיקר מסוג הנסיעה, איכות השירות הכוללת, שירות ה־wifi בטיסה, סוג הלקוח ורמת הניקיון. העובדה ש־`Total_Service_Score` מופיע כאחד המשתנים החשובים מחזקת את ההשערה שחוויית השירות הכוללת היא גורם מרכזי בחיזוי שביעות הרצון.

</div>


<div dir="rtl" style="text-align: right;">

## 9. רשתות נוירונים / MLP

בסעיף זה נבחן מודל רשת נוירונים מסוג MLP לסיווג שביעות רצון נוסעים. תחילה נמפה את משתנה המטרה לערכים בינאריים ונבצע עיבוד מקדים בתוך Pipeline כדי למנוע דליפת מידע. לאחר מכן נאמן מודל MLP בסיסי, נבצע כיוונון ארכיטקטורות באמצעות GridSearchCV, ונשווה בין מודל ברירת המחדל לבין המודל שנבחר.

</div>


<div dir="rtl" style="text-align: right;">

### 9.1 הכנת הנתונים לרשת נוירונים

לפני אימון הרשת מיפינו את משתנה המטרה לערכים בינאריים: `neutral or dissatisfied` ל-0 ו-`satisfied` ל-1. מאחר שרשת נוירונים מקבלת קלט מספרי בלבד, המשתנים הקטגוריאליים מקודדים באמצעות OneHotEncoder.

המשתנים המספריים עוברים השלמת חסרים לפי חציון ולאחר מכן StandardScaler. המשתנים הקטגוריאליים עוברים השלמת חסרים לפי הערך השכיח ולאחר מכן One-Hot Encoding. הסטנדרטיזציה חשובה במיוחד ברשתות נוירונים, משום שהאימון מבוסס על עדכון משקולות הדרגתי ורגיש לסקאלות שונות של מאפיינים. כל העיבוד מתבצע בתוך Pipeline, כך שההתאמה נעשית על סט האימון בלבד ומופעלת באותו אופן על סט האימות.

בפועל, שלב העיבוד כלל 17 משתנים מספריים ו־6 משתנים קטגוריאליים. לאחר קידוד המשתנים הקטגוריאליים התקבלו 33 מאפייני קלט לרשת.
</div>


In [ ]:
import matplotlib.pyplot as plt

from IPython.display import Markdown
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET_MAP = {
    "neutral or dissatisfied": 0,
    "satisfied": 1,
}

CLASS_LABELS_MLP = [0, 1]
CLASS_NAMES_MLP = ["Neutral/Dissatisfied", "Satisfied"]
MLP_SCORING = "accuracy"

y_train_mlp = y_train.map(TARGET_MAP)
y_valid_mlp = y_valid.map(TARGET_MAP)

if y_train_mlp.isna().any() or y_valid_mlp.isna().any():
    raise ValueError("The target contains values that are not defined in the binary mapping.")

y_train_mlp = y_train_mlp.astype(int)
y_valid_mlp = y_valid_mlp.astype(int)

target_mapping_display = pd.DataFrame(
    {
        "Original Value": list(TARGET_MAP.keys()),
        "Binary Value": list(TARGET_MAP.values()),
    }
)

display(target_mapping_display)

In [ ]:
numeric_features_mlp = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_mlp = [column for column in X_train.columns if column not in numeric_features_mlp]


def make_mlp_preprocessor():
    try:
        one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", one_hot_encoder),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features_mlp),
            ("categorical", categorical_transformer, categorical_features_mlp),
        ],
        remainder="drop",
    )


def build_mlp_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocess", make_mlp_preprocessor()),
            ("model", model),
        ]
    )


def evaluate_mlp_model(model_name, fitted_model):
    rows = []
    for split_name, X_data, y_true in [
        ("Train", X_train, y_train_mlp),
        ("Validation", X_valid, y_valid_mlp),
    ]:
        y_pred = fitted_model.predict(X_data)
        rows.append(
            {
                "Model": model_name,
                "Set": split_name,
                "Accuracy": accuracy_score(y_true, y_pred),
                "Precision": precision_score(y_true, y_pred, zero_division=0),
                "Recall": recall_score(y_true, y_pred, zero_division=0),
                "F1": f1_score(y_true, y_pred, zero_division=0),
            }
        )
    return pd.DataFrame(rows).round(4)


def get_mlp_input_size(fitted_pipeline):
    # Use transform shape instead of feature names for compatibility across sklearn versions.
    return fitted_pipeline.named_steps["preprocess"].transform(X_train.iloc[:1]).shape[1]


def normalize_hidden_layers(hidden_layer_sizes):
    if isinstance(hidden_layer_sizes, int):
        return (hidden_layer_sizes,)
    return tuple(hidden_layer_sizes)


def describe_mlp_configuration(fitted_pipeline):
    hidden_layers = normalize_hidden_layers(
        fitted_pipeline.named_steps["model"].hidden_layer_sizes
    )

    return pd.DataFrame(
        {
            "Component": [
                "Input Layer Neurons",
                "Number of Hidden Layers",
                "Neurons per Hidden Layer",
                "Total Hidden Neurons",
                "Output Layer",
            ],
            "Value": [
                get_mlp_input_size(fitted_pipeline),
                len(hidden_layers),
                str(hidden_layers),
                sum(hidden_layers),
                "Binary classification: 0 or 1",
            ],
        }
    )

mlp_preprocessing_summary = pd.DataFrame(
    {
        "Variable Type": ["Numeric", "Categorical"],
        "Number of Variables": [len(numeric_features_mlp), len(categorical_features_mlp)],
        "Preprocessing Operation": [
            "SimpleImputer(strategy='median') + StandardScaler",
            "SimpleImputer(strategy='most_frequent') + One-Hot Encoding",
        ],
    }
)

display(mlp_preprocessing_summary)

<div dir="rtl" style="text-align: right;">

### 9.2 רשת נוירונים בערכי ברירת מחדל

נריץ תחילה MLP בערכי ברירת המחדל של sklearn כדי לקבל נקודת בסיס להשוואה. לאחר העיבוד המקדים מתקבלים 33 מאפייני קלט, ולכן שכבת הקלט כוללת 33 נוירונים. ברירת המחדל של MLPClassifier כוללת שכבה חבויה אחת עם 100 נוירונים, ושכבת הפלט מבצעת סיווג בינארי בין שתי מחלקות שביעות הרצון.

במודל ברירת המחדל התקבלו על סט האימון Accuracy=0.9933, Precision=0.9946, Recall=0.9901 ו-F1=0.9923. על סט האימות התקבלו Accuracy=0.8822, Precision=0.8747, Recall=0.8524 ו-F1=0.8634. הפער הגדול בין האימון לאימות, כ-0.1111 ב-Accuracy, מעיד על התאמת יתר: המודל מתאים היטב מאוד לנתוני האימון, אך מכליל פחות טוב לנתונים שלא שימשו לאימון.

בדוח הסיווג של סט האימות התקבל F1=0.90 עבור מחלקת `Neutral/Dissatisfied` ו־F1=0.86 עבור מחלקת `Satisfied`. הפער בין המחלקות מעיד כי מודל ברירת המחדל מזהה טוב יותר את המחלקה הדומיננטית יותר בנתונים, בעוד שזיהוי מחלקת הנוסעים המרוצים מעט חלש יותר.

</div>


In [ ]:
default_mlp_pipeline = build_mlp_pipeline(
    MLPClassifier(random_state=RANDOM_STATE, max_iter=500)
)

default_mlp_pipeline.fit(X_train, y_train_mlp)

default_mlp_metrics = evaluate_mlp_model(
    "Default MLP",
    default_mlp_pipeline,
)

display(default_mlp_metrics)

print("Validation classification report for the default MLP:")
print(
    classification_report(
        y_valid_mlp,
        default_mlp_pipeline.predict(X_valid),
        labels=CLASS_LABELS_MLP,
        target_names=CLASS_NAMES_MLP,
        zero_division=0,
    )
)

default_mlp_architecture = describe_mlp_configuration(default_mlp_pipeline)

display(default_mlp_architecture)

<div dir="rtl" style="text-align: right;">

### 9.3 השוואת ארכיטקטורות לרשת הנוירונים
כיוונון ההיפר־פרמטרים בוצע באמצעות GridSearchCV, במטרה להשוות באופן שיטתי בין קומבינציות מוגדרות מראש. החיפוש התמקד בעיקר בארכיטקטורת הרשת, באמצעות השוואה בין רשתות עם שכבה חבויה אחת, שתי שכבות חבויות ושלוש שכבות חבויות. הארכיטקטורות החד־שכבתיות `(25,)`, `(50,)`, `(100,)`, `(150,)` נכללו כדי לבחון האם רשת רדודה יותר, עם מספר שונה של נוירונים, יכולה להגיע לביצועים דומים לארכיטקטורות עמוקות יותר.

ההיפר־פרמטר `hidden_layer_sizes` קובע את מספר השכבות החבויות ואת מספר הנוירונים בכל שכבה. הגדלת מספר השכבות או הנוירונים מגדילה את מורכבות המודל ועשויה לשפר את יכולתו ללמוד דפוסים מורכבים, אך גם מגדילה את זמן הריצה ואת הסיכון להתאמת יתר. לעומת זאת, רשת קטנה מדי עלולה להיות מוגבלת ביכולת הלמידה שלה.

בנוסף נבחן ההיפר־פרמטר `alpha`, השולט בעוצמת הרגולריזציה של המודל. ערך גבוה יותר מגביל את משקלי הרשת ועשוי לצמצם התאמת יתר, בעוד שערך נמוך יותר מאפשר למודל גמישות רבה יותר. פונקציית ההפעלה נקבעה ל־`relu`, המאפשרת למודל ללמוד קשרים לא ליניאריים, וקצב הלמידה ההתחלתי נקבע ל־`learning_rate_init=0.001`.

לפי תוצאות הכיוונון, הקונפיגורציה הטובה ביותר הייתה רשת עם שתי שכבות חבויות בגודל `(150, 75)`, עם סך של 225 נוירונים חבויים, `alpha=0.0001`, פונקציית הפעלה `relu` וקצב למידה התחלתי `0.001`. קונפיגורציה זו השיגה Accuracy ממוצע ב־Cross Validation של 0.8929, הגבוה ביותר מבין הקונפיגורציות שנבדקו.

עם זאת, הפערים בין חלק מהקונפיגורציות המובילות היו קטנים יחסית. לכן, הבחירה בארכיטקטורה `(150, 75)` מבוססת על יתרון ביצועים מתון, ולא על פער חד־משמעי גדול ביחס לכל שאר המודלים.
</div>


In [ ]:
architecture_options_mlp = [
    (25,),
    (50,),
    (100,),
    (150,),
    (50, 25),
    (100, 50),
    (150, 75),
    (200, 100),
    (50, 25, 10),
    (100, 50, 25),
]

mlp_search_pipeline = build_mlp_pipeline(
    MLPClassifier(
        random_state=RANDOM_STATE,
        max_iter=300,
        early_stopping=True,
        n_iter_no_change=10,
    )
)

param_grid_mlp = {
    "model__hidden_layer_sizes": architecture_options_mlp,
    "model__activation": ["relu"],
    "model__alpha": [0.0001, 0.001],
    "model__learning_rate_init": [0.001],
}

mlp_search = GridSearchCV(
    estimator=mlp_search_pipeline,
    param_grid=param_grid_mlp,
    cv=3,
    scoring=MLP_SCORING,
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)

mlp_search.fit(X_train, y_train_mlp)

mlp_tuning_results = pd.DataFrame(mlp_search.cv_results_).copy()
mlp_tuning_results["hidden_layer_sizes"] = mlp_tuning_results[
    "param_model__hidden_layer_sizes"
].apply(normalize_hidden_layers)
mlp_tuning_results["number_of_hidden_layers"] = mlp_tuning_results[
    "hidden_layer_sizes"
].apply(len)
mlp_tuning_results["total_hidden_neurons"] = mlp_tuning_results[
    "hidden_layer_sizes"
].apply(sum)
mlp_tuning_results["architecture_order"] = mlp_tuning_results[
    "hidden_layer_sizes"
].apply(architecture_options_mlp.index)

mlp_tuning_table = (
    mlp_tuning_results[
        [
            "hidden_layer_sizes",
            "number_of_hidden_layers",
            "total_hidden_neurons",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "param_model__alpha",
            "param_model__learning_rate_init",
            "param_model__activation",
            "mean_train_score",
            "architecture_order",
        ]
    ]
    .rename(
        columns={
            "hidden_layer_sizes": "hidden_layer_sizes",
            "number_of_hidden_layers": "number of hidden layers",
            "total_hidden_neurons": "total hidden neurons",
            "mean_test_score": "mean CV accuracy",
            "std_test_score": "std CV accuracy",
            "rank_test_score": "rank_test_score",
            "param_model__alpha": "alpha",
            "param_model__learning_rate_init": "learning_rate_init",
            "param_model__activation": "activation",
            "mean_train_score": "mean train accuracy",
        }
    )
    .sort_values(["rank_test_score", "architecture_order", "alpha"])
    .reset_index(drop=True)
)

display(mlp_tuning_table.drop(columns=["architecture_order"]).round(4))

In [ ]:
def get_architecture_group(hidden_layer_sizes):
    if len(hidden_layer_sizes) == 3:
        return "Three hidden layers"
    if sum(hidden_layer_sizes) >= 225:
        return "Larger two hidden layers"
    return "Two hidden layers"


architecture_plot_data = (
    mlp_tuning_table.sort_values("rank_test_score")
    .drop_duplicates(subset=["hidden_layer_sizes"])
    .sort_values("architecture_order")
    .reset_index(drop=True)
)
architecture_plot_data["architecture label"] = architecture_plot_data[
    "hidden_layer_sizes"
].apply(lambda layers: "-".join(map(str, layers)))
architecture_plot_data["architecture group"] = architecture_plot_data[
    "hidden_layer_sizes"
].apply(get_architecture_group)

architecture_colors = {
    "Two hidden layers": "#4C78A8",
    "Larger two hidden layers": "#F58518",
    "Three hidden layers": "#54A24B",
}
bar_colors = architecture_plot_data["architecture group"].map(architecture_colors)

plt.figure(figsize=(10, 5))
plt.bar(
    architecture_plot_data["architecture label"],
    architecture_plot_data["mean CV accuracy"],
    color=bar_colors,
)
plt.xlabel("Hidden layer architecture")
plt.ylabel("Mean cross-validation accuracy")
plt.title("MLP Architecture Comparison")
plt.xticks(rotation=35, ha="right")
plt.ylim(
    max(0, architecture_plot_data["mean CV accuracy"].min() - 0.02),
    min(1, architecture_plot_data["mean CV accuracy"].max() + 0.02),
)
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=color, label=label)
    for label, color in architecture_colors.items()
]
plt.legend(handles=legend_handles, title="Architecture type")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right;">

### 9.4 הרצת הרשת הנבחרת וניתוח התוצאות

המודל שנבחר כולל 33 נוירוני קלט לאחר העיבוד המקדים, שתי שכבות חבויות בגודל `(150, 75)`, פונקציית הפעלה `relu`, רגולריזציה `alpha=0.0001` וקצב למידה התחלתי `learning_rate_init=0.001`.

במודל המכוונן התקבלו על סט האימון Accuracy=0.9261, Precision=0.9218, Recall=0.9077 ו־F1=0.9147. על סט האימות התקבלו Accuracy=0.8883, Precision=0.8764, Recall=0.8664 ו־F1=0.8714.

בהשוואה למודל ברירת המחדל, שבו Accuracy האימות היה 0.8822 ו־F1 האימות היה 0.8634, הכיוונון שיפר מעט את ביצועי האימות. בנוסף, פער האימון־אימות ירד מכ־0.1111 במודל ברירת המחדל לכ־0.0378 במודל המכוונן, ולכן המודל הנבחר מציג הכללה טובה יותר ופחות התאמת יתר.


</div>


In [ ]:
best_mlp_pipeline = mlp_search.best_estimator_
best_mlp_params = mlp_search.best_params_
best_hidden_layers = normalize_hidden_layers(best_mlp_params["model__hidden_layer_sizes"])

best_mlp_metrics = evaluate_mlp_model(
    "Tuned MLP",
    best_mlp_pipeline,
)

best_mlp_params_table = pd.DataFrame(
    {
        "Parameter": [
            "hidden_layer_sizes",
            "number of hidden layers",
            "total hidden neurons",
            "activation",
            "alpha",
            "learning_rate_init",
            "best mean CV accuracy",
        ],
        "Selected Value": [
            str(best_hidden_layers),
            len(best_hidden_layers),
            sum(best_hidden_layers),
            best_mlp_params["model__activation"],
            best_mlp_params["model__alpha"],
            best_mlp_params["model__learning_rate_init"],
            round(mlp_search.best_score_, 4),
        ],
    }
)

best_mlp_architecture = describe_mlp_configuration(best_mlp_pipeline)

display(best_mlp_params_table)
display(best_mlp_architecture)
display(best_mlp_metrics)

In [ ]:
best_mlp_valid_predictions = best_mlp_pipeline.predict(X_valid)

print("Validation classification report for the selected MLP:")
print(
    classification_report(
        y_valid_mlp,
        best_mlp_valid_predictions,
        labels=CLASS_LABELS_MLP,
        target_names=CLASS_NAMES_MLP,
        zero_division=0,
    )
)


ConfusionMatrixDisplay.from_predictions(
    y_valid_mlp,
    best_mlp_valid_predictions,
    display_labels=["Neutral/Dissatisfied", "Satisfied"],
    cmap="Blues",
    values_format="d",
)
plt.title("Tuned MLP Confusion Matrix")
plt.show()

<div dir="rtl" style="text-align: right;">

מטריצת הבלבול של המודל המכוונן מראה כי 918 תצפיות מהמחלקה `Neutral/Dissatisfied` סווגו נכון, ו־681 תצפיות מהמחלקה `Satisfied` סווגו נכון. בנוסף, 96 תצפיות שאינן מרוצות או ניטרליות סווגו בטעות כמרוצות, ו־105 תצפיות מרוצות סווגו בטעות כלא מרוצות או ניטרליות.

תוצאות אלו מצביעות על כך שהמודל מצליח לסווג את מרבית התצפיות בשתי המחלקות, אך עדיין קיימות טעויות בשני הכיוונים. שיעור הטעויות דומה יחסית בין המחלקות, ולכן לא נראית הטיה חריפה למחלקה אחת בלבד.

</div>

In [ ]:
default_valid_accuracy = default_mlp_metrics.loc[
    default_mlp_metrics["Set"] == "Validation", "Accuracy"
].iloc[0]
best_train_accuracy = best_mlp_metrics.loc[
    best_mlp_metrics["Set"] == "Train", "Accuracy"
].iloc[0]
best_valid_accuracy = best_mlp_metrics.loc[
    best_mlp_metrics["Set"] == "Validation", "Accuracy"
].iloc[0]
generalization_gap = best_train_accuracy - best_valid_accuracy

if best_valid_accuracy > default_valid_accuracy:
    mlp_conclusion = (
        f"The architecture comparison improved validation Accuracy from {default_valid_accuracy:.4f} "
        f"for the default model to {best_valid_accuracy:.4f} for the selected model. "
        f"The selected architecture is {best_hidden_layers}. "
    )
elif best_valid_accuracy < default_valid_accuracy:
    mlp_conclusion = (
        f"The default model achieved higher validation Accuracy ({default_valid_accuracy:.4f}) "
        f"than the architecture selected by the search ({best_valid_accuracy:.4f}). "
        f"The selected architecture is {best_hidden_layers}, so it is important to verify whether "
        "the additional complexity actually improves generalization. "
    )
else:
    mlp_conclusion = (
        f"The default and tuned models achieved the same validation Accuracy ({best_valid_accuracy:.4f}). "
        f"The selected architecture is {best_hidden_layers}. In this case, the simpler model can be "
        "preferred unless other metrics justify a different choice. "
    )

mlp_conclusion += (
    f"The Accuracy gap between training and validation for the selected model is {generalization_gap:.4f}. "
    "A large gap may indicate overfitting, especially in networks with more layers or neurons."
)

display(Markdown(mlp_conclusion))

<div dir="rtl" style="text-align: right;">

## 10. השוואה בין מודלים

לפני ההשוואה המספרית בין המודלים, חשוב להתייחס לשאלה מדוע למשימת הסיווג שלנו עדיף להשתמש במודלים מפוקחים כמו `Decision Tree` או `Neural Network` ולא ב-`K-Means`. בפרויקט זה קיימת לנו עמודת יעד ידועה, `satisfaction`, והמטרה היא לחזות האם נוסע יהיה מרוצה או לא מרוצה. לכן מדובר בבעיית סיווג מפוקחת: המודל לומד מדוגמאות שבהן כבר ידועה התשובה הנכונה, ואז משתמש בלמידה הזו כדי לחזות עבור נתונים חדשים.

לעומת זאת, `K-Means` הוא אלגוריתם לא מפוקח. הוא אינו משתמש בעמודת היעד בזמן האימון, אלא מחלק את הנוסעים לקבוצות לפי דמיון במאפיינים שלהם. חלוקה כזו יכולה לעזור לגלות קבוצות מעניינות של נוסעים, אך היא אינה מיועדת לחיזוי ישיר של `satisfaction`. לכן במקרה שלנו `Decision Tree` ו-`Neural Network` מתאימים יותר למשימה המרכזית.
ההשוואה המספרית בהמשך הסעיף תתבצע רק בין שני המודלים המפוקחים: עץ ההחלטה הגזום ורשת הנוירונים המכווננת. את ההשוואה נבצע לפי עקרונות ההכללה שנלמדו בהרצאה: בחירת מודל צריכה להתבסס על ביצועים בנתוני אימות, ולא רק על ביצועי האימון.

המדד הראשי להשוואה בסעיף זה הוא `Validation Accuracy`, ובנוסף נבחן גם את `Precision`, `Recall`, `F1` ואת פער ההכללה בין האימון לאימות. בסוף הסעיף נזהה מודל מוביל זמני, בעוד שהבחירה הרשמית למודל ההגשה תישמר לסעיף 11.

</div>


In [ ]:
dt_train_row = best_dt_metrics.loc[best_dt_metrics["Dataset"] == "Train"].iloc[0]
dt_valid_row = best_dt_metrics.loc[best_dt_metrics["Dataset"] == "Validation"].iloc[0]
mlp_train_row = best_mlp_metrics.loc[best_mlp_metrics["Set"] == "Train"].iloc[0]
mlp_valid_row = best_mlp_metrics.loc[best_mlp_metrics["Set"] == "Validation"].iloc[0]

dt_gap = dt_train_row["Accuracy"] - dt_valid_row["Accuracy"]
mlp_gap = mlp_train_row["Accuracy"] - mlp_valid_row["Accuracy"]

dt_configuration = (
    f"max_depth={best_combined_depth}, "
    f"ccp_alpha={best_combined_alpha:.6f}, "
    f"min_samples_split={best_combined_split}, "
    f"min_samples_leaf={best_combined_leaf}"
)

model_comparison_table = pd.DataFrame(
    [
        {
            "Model": "Decision Tree",
            "Train Accuracy": dt_train_row["Accuracy"],
            "Validation Accuracy": dt_valid_row["Accuracy"],
            "Validation Precision": dt_valid_row["Precision"],
            "Validation Recall": dt_valid_row["Recall"],
            "Validation F1": dt_valid_row["F1"],
            "Generalization Gap": dt_gap,
            "Chosen Configuration": dt_configuration,
            "Interpretability": "High",
            "Complexity": "Moderate",
        },
        {
            "Model": "MLP",
            "Train Accuracy": mlp_train_row["Accuracy"],
            "Validation Accuracy": mlp_valid_row["Accuracy"],
            "Validation Precision": mlp_valid_row["Precision"],
            "Validation Recall": mlp_valid_row["Recall"],
            "Validation F1": mlp_valid_row["F1"],
            "Generalization Gap": mlp_gap,
            "Chosen Configuration": (
                f"{best_hidden_layers}, {best_mlp_params['model__activation']}, "
                f"alpha={best_mlp_params['model__alpha']}, "
                f"learning_rate_init={best_mlp_params['model__learning_rate_init']}"
            ),
            "Interpretability": "Low",
            "Complexity": "Higher",
        },
    ]
)

display(model_comparison_table.round(4))


In [ ]:
validation_plot_data = model_comparison_table[
    [
        "Model",
        "Validation Accuracy",
        "Validation Precision",
        "Validation Recall",
        "Validation F1",
    ]
].melt(id_vars="Model", var_name="Metric", value_name="Score")

metric_label_map = {
    "Validation Accuracy": "Accuracy",
    "Validation Precision": "Precision",
    "Validation Recall": "Recall",
    "Validation F1": "F1",
}
validation_plot_data["Metric"] = validation_plot_data["Metric"].map(metric_label_map)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=validation_plot_data,
    x="Metric",
    y="Score",
    hue="Model",
    palette="muted",
)

for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(
        f"{height:.3f}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha="center",
        va="bottom",
        fontsize=9,
        xytext=(0, 3),
        textcoords="offset points",
    )

plt.title("Validation Metrics Comparison")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend(title="Model")
plt.tight_layout()
plt.show()


<div dir="rtl" style="text-align: right;">

**מסקנות מההשוואה:**

ההשוואה הכמותית בוצעה בין שני המודלים שאומנו בפועל: עץ החלטה ורשת נוירונים מסוג MLP. מהטבלה והגרף ניתן לראות כי שני המודלים מציגים ביצועים קרובים מאוד על סט האימות, ולכן אין פער חד בין החלופות.

עם זאת, עץ ההחלטה מציג יתרון קל ברוב מדדי האימות: דיוק של 0.8917 לעומת 0.8883 ב־MLP, Precision של 0.8863 לעומת 0.8764, ו־F1 של 0.8743 לעומת 0.8714. לעומת זאת, ה־MLP מציג Recall מעט גבוה יותר, 0.8664 לעומת 0.8626, כלומר הוא מזהה מעט יותר מקרים חיוביים, אך היתרון במדד זה קטן יחסית.

בנוסף, פער ההכללה של עץ ההחלטה נמוך יותר: 0.0119 לעומת 0.0378 ב־MLP. כלומר, ה־MLP מגיע לביצועים גבוהים יותר על סט האימון, אך הפער הגדול יותר מול סט האימות עשוי להעיד על התאמה חזקה יותר לנתוני האימון. לכן, בהתאם לעקרונות ההכללה, בחירת המודל אינה מתבססת רק על ביצועי האימון, אלא בעיקר על ביצועי האימות, פער ההכללה ומורכבות המודל.

לאור זאת, למרות שה־MLP מציג ביצועים תחרותיים מאוד, המודל שנבחר להמשך הוא עץ ההחלטה. הבחירה בו נובעת מהשילוב בין ביצועי אימות מעט טובים יותר, פער הכללה קטן יותר, ומידת פרשנות גבוהה יותר המאפשרת להסביר אילו משתנים תרמו לסיווג.



</div>


<div dir="rtl" style="text-align: right;">

## 11. המודל הנבחר

בהתבסס על השוואת ביצועי המודלים על סט האימות, המודל שנבחר להגשה הוא עץ ההחלטה המיטבי.

הקונפיגורציה שנבחרה עבור המודל היא:
`max_depth=9`, `ccp_alpha=0`, `min_samples_split=30`, `min_samples_leaf=25`.

הבחירה בעץ ההחלטה נעשתה משום שהוא השיג את ה־Validation Accuracy הגבוה ביותר מבין המודלים שנבחנו: 0.8917 לעומת 0.8883 ב־MLP. בנוסף, הוא השיג F1 גבוה יותר ופער הכללה קטן יותר, ולכן נראה שהוא יציב יותר על נתונים שלא שימשו לאימון. מעבר לכך, לעץ החלטה יש יתרון פרשני ברור: ניתן להציג את מבנה העץ, לעקוב אחר מסלול החלטה של רשומה בודדת, ולנתח את חשיבות המשתנים.

</div>

In [ ]:
# ציור מטריצת בלבול עבור המודל הנבחר על סט האימות
best_dt_valid_predictions = best_dt_pipeline.predict(X_valid)

plt.figure(figsize=(6, 5), dpi=150)
ConfusionMatrixDisplay.from_predictions(
    y_valid_dt,
    best_dt_valid_predictions,
    display_labels=["Neutral/Dissatisfied", "Satisfied"],
    cmap="Blues",
    values_format="d"
)
plt.title("Selected Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right;">

מטריצת הבלבול של עץ ההחלטה על סט האימות מראה כי המודל סיווג נכון 927 תצפיות מהמחלקה `Neutral/Dissatisfied` ו־678 תצפיות מהמחלקה `Satisfied`. בנוסף, 87 תצפיות שאינן מרוצות/ניטרליות סווגו בטעות כמרוצות, ו־108 תצפיות מרוצות סווגו בטעות כלא מרוצות/ניטרליות.

מהמטריצה ניתן לראות שאין קריסה למחלקה אחת בלבד, ולכן לא נראה שהמודל מוטה באופן חריף לאחת המחלקות. עם זאת, קיימת חולשה מסוימת בזיהוי נוסעים מרוצים, שכן 108 מהם סווגו בטעות כלא מרוצים/ניטרליים.

</div>

<div dir="rtl" style="text-align: right;">

## 12. חיזויים סופיים וייצוא קובץ ההגשה

בסעיף זה נאמן מחדש את עץ ההחלטה שנבחר על כלל נתוני האימון הנקיים, נחזה את התוויות עבור קובץ המבחן הסופי ונייצא קובץ Excel בפורמט הנדרש להגשה.

</div>


In [ ]:
# מיפוי משתנה המטרה המלא לערכים הבינאריים ששימשו באימון המודלים
y_full_dt = y_model.map(TARGET_MAP)

if y_full_dt.isna().any():
    raise ValueError("The full training target contains values that are not defined in TARGET_MAP.")

y_full_dt = y_full_dt.astype(int)

# קובץ הדוגמה קובע את מבנה העמודות בלבד; מספר השורות נקבע לפי קובץ המבחן
expected_submission_columns = list(example_submission.columns)
if expected_submission_columns != ["target"]:
    raise ValueError(
        f"Unexpected sample submission structure: {expected_submission_columns}. "
        "Expected exactly one column named 'target'."
    )

final_test_index_before_prediction = X_test_final.index.copy()
final_test_rows_before_prediction = len(X_test_final)

# בניית Pipeline חדש עם הקונפיגורציה שנבחרה ואימון על כל נתוני האימון הנקיים
final_dt_model = DecisionTreeClassifier(
    max_depth=best_combined_depth,
    ccp_alpha=best_combined_alpha,
    min_samples_split=best_combined_split,
    min_samples_leaf=best_combined_leaf,
    random_state=RANDOM_STATE,
)

final_dt_pipeline = build_dt_pipeline(final_dt_model)
final_dt_pipeline.fit(X_model, y_full_dt)

# חיזוי לפי הסדר המקורי של קובץ המבחן
final_test_predictions = final_dt_pipeline.predict(X_test_final).astype(int)
submission = pd.DataFrame(
    {"target": final_test_predictions},
    index=X_test_final.index,
)

# בדיקות תקינות לפני הייצוא
assert len(X_test_final) == final_test_rows_before_prediction
assert X_test_final.index.equals(final_test_index_before_prediction)
assert len(submission) == len(X_test_final) == len(test_raw)
assert submission.index.equals(X_test_final.index)
assert list(submission.columns) == expected_submission_columns == ["target"]
assert submission["target"].isna().sum() == 0
assert set(submission["target"].unique()).issubset({0, 1})

SUBMISSION_FILE = BASE_DIR / "airline_G3_ytest.xlsx"
submission.to_excel(SUBMISSION_FILE, index=False)

# קריאה חוזרת מהקובץ כדי לוודא שהייצוא עצמו תקין
exported_submission = pd.read_excel(SUBMISSION_FILE)
assert len(exported_submission) == len(X_test_final)
assert list(exported_submission.columns) == ["target"]
assert exported_submission["target"].isna().sum() == 0
assert set(exported_submission["target"].unique()).issubset({0, 1})
pd.testing.assert_frame_equal(
    submission.reset_index(drop=True),
    exported_submission.reset_index(drop=True),
    check_dtype=False,
)

submission_summary = pd.DataFrame(
    {
        "Check": [
            "Output file",
            "Training rows used",
            "Prediction rows",
            "Final test row count preserved",
            "Final test row order preserved",
            "Missing predictions",
            "Valid binary predictions",
            "Exported file matches in-memory submission",
        ],
        "Result": [
            SUBMISSION_FILE.name,
            len(X_model),
            len(submission),
            len(X_test_final) == len(test_raw),
            X_test_final.index.equals(test_raw.index),
            int(submission["target"].isna().sum()),
            set(submission["target"].unique()).issubset({0, 1}),
            submission.reset_index(drop=True).equals(exported_submission.reset_index(drop=True)),
        ],
    }
)

prediction_distribution = (
    submission["target"]
    .value_counts()
    .sort_index()
    .rename_axis("Predicted Class")
    .reset_index(name="Number of Predictions")
)
prediction_distribution["Percentage"] = (
    prediction_distribution["Number of Predictions"] / len(submission) * 100
).round(2)

display(submission_summary)
display(prediction_distribution)
display(submission.head())

print(f"Submission file created: {SUBMISSION_FILE.name}")
print(f"Submission path: {SUBMISSION_FILE.resolve()}")
print("All submission validation checks passed.")

<div dir="rtl" style="text-align: right;">

לאחר בחירת עץ ההחלטה כמודל הסופי, נבנה Pipeline חדש עם הקונפיגורציה שנבחרה בתהליך הכוונון. המודל אומן מחדש על כלל נתוני האימון הנקיים, כולל הרשומות ששימשו קודם לכן כסט אימות, משום שתהליך בחירת המודל כבר הסתיים.

קובץ המבחן הסופי עבר את אותו תהליך ניקוי ועיבוד מקדים שהוגדר במחברת. לא נמחקו ממנו רשומות, מספר הרשומות וסדרן המקורי נשמרו, והוא לא שימש בשום שלב לבחירת המודל או לכוונון ההיפר־פרמטרים.

החיזויים יוצאו לקובץ `airline_G3_ytest.xlsx` הכולל עמודה יחידה בשם `target`. לפני סיום התהליך נבדק כי הקובץ מכיל מספר חיזויים זהה למספר הרשומות בקובץ המבחן, שאין בו ערכים חסרים, שכל החיזויים הם 0 או 1, ושהקובץ שנשמר ב־Excel זהה לטבלת החיזויים שנוצרה בזיכרון.

</div>